In [3]:
import pandas as pd

df = pd.read_parquet("hf://datasets/to-be/epstein-emails/emails.parquet")

In [3]:
df.shape

(4272, 12)

In [4]:
df.columns

Index(['id', 'email_document_id', 'source_filename', 'subject',
       'message_order', 'from_address', 'to_address', 'other_recipients',
       'timestamp_raw', 'timestamp_iso', 'message_html', 'document_id'],
      dtype='object')

Converter coluna 'from_address' e 'to_address' para strings, caso já não sejam.

In [5]:
df['from_address'] = df['from_address'].fillna('').astype(str) # NAN caso vazio
df['to_address']   = df['to_address'].fillna('').astype(str) 

In [6]:
df[['from_address', 'to_address']].isna().sum()

from_address    0
to_address      0
dtype: int64

Converter 'other_recipients' para um vetor do tipo [obj_1, obj_2, ..., obj_n].

In [8]:
import ast

df['other_recipients'] = (
    df['other_recipients']
    .fillna('[]')
    .apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
)

Agora vamos expandir 'other_recipients' em linhas copiadas individuais e também 'to_address' caso exista endereços múltiplos.

Agora fazemos a expansão.

In [9]:
import re

# 1) Garante que other_recipients é lista
df['other_recipients'] = df['other_recipients'].apply(lambda x: x if isinstance(x, list) else [])

# 2) Converte to_address em lista quando tiver múltiplos
def split_to_address(x):
    if not isinstance(x, str):
        return []
    partes = re.split(r'[;,]', x)
    return [p.strip() for p in partes if p.strip()]

df['to_address_list'] = df['to_address'].apply(split_to_address)

# 3) Junta to_address + other_recipients num único vetor
df['todos_destinatarios'] = df.apply(
    lambda row: row['to_address_list'] + row['other_recipients'],
    axis=1
)

# 4) Explode tudo em linhas individuais
df_exp = df.copy()
df_exp['destinatario'] = df_exp['todos_destinatarios']
df_exp = df_exp.explode('destinatario')

# 5) Substitui to_address e limpa other_recipients
df_exp['to_address'] = df_exp['destinatario']
df_exp['other_recipients'] = [[] for _ in range(len(df_exp))]

# 6) Remove colunas auxiliares
df_final = df_exp.drop(columns=['to_address_list', 'todos_destinatarios', 'destinatario'])

In [14]:
df_final.shape

(5022, 12)

In [13]:
df_final['other_recipients'].apply(len).eq(0).sum()

np.int64(5022)

In [15]:
df_final.head()

,id,email_document_id,source_filename,subject,message_order,from_address,to_address,other_recipients,timestamp_raw,timestamp_iso,message_html,document_id
0,1,1,HOUSE_OVERSIGHT_012102.txt,Email subject of the thread,0,Jeffrey Epstein,Nicholas Ribis,[],5/7/2019 4:29:00 AM,20190507042900,<a href='https://www.nytimes.com/2019/05/06/us...,HOUSE_OVERSIGHT_012102
1,2,2,HOUSE_OVERSIGHT_012898.txt,Farmer Jaffe is suing Donald Trump!,0,Tonja Haddad Coleman,Jeffrey Epstein,[],5/13/2013 10:28:30 PM,20130513222830,Farmer Jaffe is suing Donald Trump!<br><a href...,HOUSE_OVERSIGHT_012898
1,2,2,HOUSE_OVERSIGHT_012898.txt,Farmer Jaffe is suing Donald Trump!,0,Tonja Haddad Coleman,Darren Indyke,[],5/13/2013 10:28:30 PM,20130513222830,Farmer Jaffe is suing Donald Trump!<br><a href...,HOUSE_OVERSIGHT_012898
1,2,2,HOUSE_OVERSIGHT_012898.txt,Farmer Jaffe is suing Donald Trump!,0,Tonja Haddad Coleman,Debbie Fein,[],5/13/2013 10:28:30 PM,20130513222830,Farmer Jaffe is suing Donald Trump!<br><a href...,HOUSE_OVERSIGHT_012898
2,3,3,HOUSE_OVERSIGHT_014516.txt,Fwd: 2016 Election: Tax Changes Expected,0,Richard Kahn,Jeffrey Epstein,[],12/15/2016 5:40:04 PM,20161215174004,,HOUSE_OVERSIGHT_014516


Vamos apagar as duplicatas:

In [16]:
df_final = df_final.drop_duplicates(
    subset=[
        'id', 'email_document_id', 'source_filename', 'subject',
        'message_order', 'from_address', 'to_address',
        'timestamp_raw', 'timestamp_iso', 'message_html', 'document_id'
    ],
    keep='first'
)

Criando csv com três colunas: 'de', 'para' e 'assunto'. 

In [17]:
def criar_csv_de_para_assunto(df_final):
    df_saida = df_final[['from_address', 'to_address', 'subject']].copy()
    df_saida.columns = ['de', 'para', 'assunto']
    return df_saida

df_saida = criar_csv_de_para_assunto(df_final)

In [18]:
df_saida.head(15)

,de,para,assunto
0,Jeffrey Epstein,Nicholas Ribis,Email subject of the thread
1,Tonja Haddad Coleman,Jeffrey Epstein,Farmer Jaffe is suing Donald Trump!
1,Tonja Haddad Coleman,Darren Indyke,Farmer Jaffe is suing Donald Trump!
1,Tonja Haddad Coleman,Debbie Fein,Farmer Jaffe is suing Donald Trump!
2,Richard Kahn,Jeffrey Epstein,Fwd: 2016 Election: Tax Changes Expected
3,"Ens, Amanda",Richard Kahn,Fwd: 2016 Election: Tax Changes Expected
4,Nicholas Ribis,Jeffrey Epstein,RE:
5,John Page,Mayor,Russian House
6,<REDACTED>,John Page,Russian House
7,"Ens, Amanda",Jeffrey Epstein,SPX put contingent on higher rates


Agora vamos ver os valores únicos e dropar os valores inúteis.

In [21]:
def limpar_nome(x):
    if not isinstance(x, str):
        return x
    
    # Remove emails entre < >.
    x = re.sub(r'<[^>]+>', '', x)
    
    # Remove emails soltos.
    x = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '', x)
    
    # Remove aspas, colchetes, parênteses.
    x = re.sub(r'["\[\]\(\)]', '', x)
    
    # Remove múltiplos espaços.
    x = re.sub(r'\s+', ' ', x).strip()
    
    return x

Aplicando:

In [22]:
df_saida['de'] = df_saida['de'].apply(limpar_nome)
df_saida['para'] = df_saida['para'].apply(limpar_nome)

Vamos remover linhas que ficaram com 'de' ou 'para' vazios.

In [23]:
df_saida = df_saida[
    df_saida['de'].notna() & df_saida['para'].notna() &
    (df_saida['de'].str.strip() != '') &
    (df_saida['para'].str.strip() != '')
]

In [24]:
print("Linhas restantes:", len(df_saida))

Linhas restantes: 4825


Agora vamos atacar os valores únicos, pois provavelmente tem valores que se referem as mesmas pessoas sendo vistas de maneira distintas.

In [25]:
pessoas_de = df_saida['de'].unique()
pessoas_de = sorted(pessoas_de)
for pessoa in pessoas_de:
    print(pessoa)

-11>
-1M1=1111M1>
111
1=1=11111M11
<1111111
<1=1=11111M11
<>
>
????
A Barrett
Afri zp
Al Franken
Al seckel
Al seckel 111111111111111.11111111111111111
Alan
Alan Dershowitz
Alan Rogers
Alexandra Preate
Alireza ITTIHADIEH
Alireza Ittihadieh
Anas Alrasheed
Anders Corr
Andres Serrano
Andrew Friendly
Anil Ambani
Anil.Ambani
Ann Coulter
Annette Witheridge
Ariane de Rothschild
Ashley Parker
Auren Hoffman
BARBRO EHNBOM
BRYAN SUBOTNICK
BRYAN SUBOTNICK 411111111111>
Barbro C Ehnbom
Barbro C. Ehnbom
Barbro Ehnbom
Barnaby Marsh
Barry J. Cohen
Barry J. Cohen <>
Barry Josephson
Benjamin Harnwell
Benjamin Harnwell <
Benjamin Harnwell <>
Benjamin Wegg-Prosser
Bill Siegel
Blanche Christerso
Bloomberg
Bob Crowe
Boris Nikoli
Boris Nikolic
Brad Edwards
Brad Wechsler <>
Brandon Thompson
Browning, Jack mailto
Bruce Moskowitz
CLL
CNS Adam Klasfeld
Carolyn Rangel
Cathy Alexander
Cecilia Schott
Cecilia Schött
Cecilia Steen
Christina Galbraith
Comms Alert
DAN CHRISTENSEN
DAVID SCHOEN
Dangene and Jennie Enterpri

Aplicando mapeamento para limpeza e padronização da coluna 'de'.

In [27]:
mapeamento = {
    "-11>": "Desconhecido 1",
    "-1M1=1111M1>": "Desconhecido 2",
    "111": "Desconhecido 3",
    "1=1=11111M11": "Desconhecido 4",
    "<1111111": "Desconhecido 5",
    "<1=1=11111M11": "Desconhecido 6",
    "<>": "Desconhecido 7",
    ">": "Desconhecido 8",
    "????": "Desconhecido 9",
    "Al seckel 111111111111111.11111111111111111": "Al seckel",
    "Alan": "Alan Dershowitz",
    "Alireza ITTIHADIEH": "Alireza Ittihadieh",
    "Anil.Ambani": "Anil Ambani",
    "anil.ambani": "Anil Ambani",
    "BARBRO EHNBOM": "Barbro C. Ehnbom",
    "Barbro C Ehnbom": "Barbro C. Ehnbom",
    "Barbro Ehnbom": "Barbro C. Ehnbom",
    "BRYAN SUBOTNICK": "Bryan Subotnick",
    "BRYAN SUBOTNICK 411111111111>": "Bryan Subotnick",
    "Barry J. Cohen <>": "Barry J. Cohen",
    "Barry Josephson": "Barry J. Cohen",
    "Benjamin Harnwell <": "Benjamin Harnwell",
    "Benjamin Harnwell <>": "Benjamin Harnwell",
    "Boris Nikoli": "Boris Nikolic",
    "Brad Wechsler <>": "Brad Wechsler",
    "Browning, Jack mailto": "Jack Browning",
    "Cecilia Schott": "Cecilia Schött",
    "DAN CHRISTENSEN": "Dan Christensen",
    "DAVID SCHOEN": "David Schoen",
    "Daniel Sabba I": "Daniel Sabba",
    "Darren Indyke <________________": "Darren K. Indyke",
    "Darren Indyke": "Darren K. Indyke",
    "David": "David Bianco, Deutsche Bank",
    "David Schoen <>": "David Schoen",
    "Deepak Chopra <>": "Deepak Chopra",
    "Deepak Chopra <__________________________": "Deepak Chopra",
    "Deepak Chopra____________": "Deepak Chopra",
    "Diane Ziman <>": "Diane Ziman",
    "Dr. Bruce L. Dennis <>": "Dr. Bruce L. Dennis",
    "EL HACHEM Johnny": "Johnny el Hachem",
    "Ed Boyden <>": "Ed Boyden",
    "Ed Boyden <______________________": "Ed Boyden",
    "Ed": "Ed Boyden",
    "Edyta.Wynberg": "Edyta Wynberg",
    "Eisenberg, Marshall E.": "Marshall E. Eisenberg",
    "Ens, Amand": "Amanda Ens",
    "Ens, Amanda": "Amanda Ens",
    "Eric Roth <>": "Eric Roth",
    "Eva pichand": "Eva Pichand",
    "Faith Kates <>": "Faith Kates",
    "Faith Kate": "Faith Kates",
    "Fareed Zakaria GPS": "Fareed Zakaria",
    "Fred Haddad <>": "Fred Haddad",
    "G Maxwell": "Ghislaine Maxwell",
    "GMAX": "Ghislaine Maxwell",
    "Gmax": "Ghislaine Maxwell",
    "gmax": "Ghislaine Maxwell",
    "Gi Feather on behalf of Gerald Barton": "Gerald Barton",
    "Gonzalez, Patricia": "Patricia Gonzalez",
    "Greg": "Greg Farrell",
    "Greg Farrell BLOOMBERG/ NEWSROOM:": "Greg Farrell",
    "Guy Amico <>": "Guy Amico",
    "Gwendolyn": "Gwendolyn Beck",
    "HOUSE OVERSIGHT": "House Oversight",
    "HOUSE OVERSIGHT 026675": "House Oversight",
    "HOUSE OVERSIGHT 032374": "House Oversight",
    "HOUSE OVERSIGHT 032605": "House Oversight",
    "HOUSE OVERSIGHT 033178": "House Oversight",
    "HOUSE OVERSIGHT 033246": "House Oversight",
    "HOUSE OVERSIGHT 033292": "House Oversight",
    "HOUSE OVERSIGHT 033309": "House Oversight",
    "HOUSE_OVERSIGHT_029969": "House Oversight",
    "HOUSE_OVERSIGHT_030630": "House Oversight",
    "HOUSE_OVERSIGHT_030807": "House Oversight",
    "HOUSE_OVERSIGHT_030874": "House Oversight",
    "Haig, David": "David Haig",
    "Hayes, David L., M.D.": "Dr. David L. Hayes",
    "Heather Mann": "Heather Man",
    "Heberlig, Brian": "Brian Heberlig",
    "Hill, James E.": "James E. Hill",
    "Hope, Doug": "Doug Hope",
    "Howard Rubenstein <>": "Howard Rubenstein",
    "IMMIIIIIMIMI": "Desconhecido 10",
    "I_____________________________": "Desconhecido 11",
    "Immelt, Stephen J.": "Stephen J. Immelt",
    "Ingram, David Reuters News": "David Ingram News",
    "J": "Desconhecido 12",
    "J Jep": "Desconhecido 12",
    "JEE": "Desconhecido 12",
    "JONATHAN FARKAS": "Jonathan Farkas",
    "JONATHAN FARKASIIII": "Jonathan Farkas",
    "Jonathan": "Jonathan Farkas",
    "Jonathan Farkailll": "Jonathan Farkas",
    "Jonathan Farkas ..111.": "Jonathan Farkas",
    "Jabor Y. INEWEI": "Jabor Y. Inewey",
    "Jabor Y": "Jabor Y. Inewey",
    "Jabor Y.": "Jabor Y. Inewey",
    "Jack LANG": "Jack Lang",
    "Jack LANG <>": "Jack Lang",
    "Jamie Rosenwald <___________________________________": "Jamie Rosenwald",
    "Jean HUGUEN": "Jean Huguen",
    "Jessica Cadwell, Paralegal": "Jessica Cadwell",
    "Joel": "Joel Dunn",
    "John Connolly Ed.D.": "John Connolly",
    "Joi Ita": "Joichi Ito",
    "Joi Ito": "Joichi Ito",
    "Joi Ito <": "Joichi Ito",
    "Joi Ito <>": "Joichi Ito",
    "Kaczorowski, Monice M.": "Monice M. Kaczorowski",
    "Karp, Brad S": "Brad S. Karp",
    "Kathy": "Kathryn Ruemmler",
    "Kathy Ruemmle": "Kathryn Ruemmler",
    "Kathy Ruemmler": "Kathryn Ruemmler",
    "Kathy Ruemmler <": "Kathryn Ruemmler",
    "Kathy Ruemmler <____________________": "Kathryn Ruemmler",
    "Kathy Ruemmler <_______________________": "Kathryn Ruemmler",
    "Kathy Ruemmler <____________________________-": "Kathryn Ruemmler",
    "Kathy Ruemmler___________________": "Kathryn Ruemmler",
    "Kathy Ruemmlerl": "Kathryn Ruemmler",
    "Katily Ruemmler imumniummu": "Kathryn Ruemmler",
    "Kelly Friendly <>": "Kelly Friendly",
    "Ken Star": "Ken Starr",
    "Ken Starr <": "Ken Starr",
    "Ken Starr <>": "Ken Starr",
    "Ken Starr <__________________": "Ken Starr",
    "Ken Starr__________________________________": "Ken Starr",
    "LH": "Lawrence H. Summers",
    "LHS": "Lawrence H. Summers",
    "LHS <": "Lawrence H. Summers",
    "LHS <>": "Lawrence H. Summers",
    "LHS___________": "Lawrence H. Summers",
    "LHS__________________________": "Lawrence H. Summers",
    "Lawrence H. Summers <>": "Lawrence H. Summers",
    "Lawrence Summers": "Lawrence H. Summers",
    "Lawrence Summers <>": "Lawrence H. Summers",
    "Landon Thomas": "Landon Thomas Jr.",
    "Landon Thomas Jr. <": "Landon Thomas Jr.",
    "Lang, Caroline": "Caroline Lang",
    "Larry": "Larry Summers",
    "Larry Summer": "Larry Summers",
    "Larry Summers -": "Larry Summers",
    "Larry Summers <": "Larry Summers",
    "Larry Summers <>": "Larry Summers",
    "Larry Visosk": "Larry Visoski",
    "Larry Visoski <>": "Larry Visoski",
    "Lawrence Krauss <>": "Lawrence Krauss",
    "Leah Reis-Dennis <1": "Leah Reis-Dennis",
    "Leon Black <IMMIMI": "Leon Black",
    "Lesley Groff <>": "Lesley Groff",
    "Lesley Groff I_____________________________": "Lesley Groff",
    "Lilly Sanchez <___________________": "Lilly Sanchez",
    "Lilly Sanchez I": "Lilly Sanchez",
    "Linda Stone <I_______________________r-": "Linda Stone",
    "Lisa Ne": "Lisa New",
    "Lisa New <1": "Lisa New",
    "Lisa New <>": "Lisa New",
    "Lisa New________________________": "Lisa New",
    "Lvje": "Las Vegas Jewish Experience",
    "Lvjet": "Las Vegas Jewish Experience",
    "MARK TRAMO": "Mark Tramo",
    "MMIIIMIM": "Desconhecido 13",
    "MOMMINNEMUMMIN": "Desconhecido 13",
    "Marc.Kensington": "Marc Kensington",
    "Martin G. Weinberg <": "Martin G. Weinberg",
    "Martin G. Weinberg <>": "Martin G. Weinberg",
    "Martin Weinberg": "Martin G. Weinberg",
    "Martin Weinberg iMiniMIM>": "Martin G. Weinberg",
    "Masha Drokova <________________": "Masha Drokova",
    "Maskin, Eric": "Eric Maskin",
    "Michael Wolff -": "Michael Wolff",
    "Michael Wolff 111111111111111.1111111111.1.1.11111111111111.": "Michael Wolff",
    "Michael Wolff <1.111'": "Michael Wolff",
    "Michael Wolff <1111": "Michael Wolff",
    "Michael Wolff <>": "Michael Wolff",
    "Michael Wolff <________________": "Michael Wolff",
    "Michael Wolff <j____________________________k": "Michael Wolff",
    "Michael Wolff MMMNIMMIMMI": "Michael Wolff",
    "Michael Wolff _______________________________": "Michael Wolff",
    "Michael Wolff______________________": "Michael Wolff",
    "Michael Wolffl": "Michael Wolff",
    "Michael, Charles": "Charles Michael",
    "Miller, Micha": "Micha Miller",
    "Miller, Michael": "Michael Miller",
    "Mohamed Waheed": "Mohammed Waheed Hassan",
    "Mohamed Waheed Hassan": "Mohammed Waheed Hassan",
    "Mosk, Matthew": "Matthew Mosk",
    "Moskowitz, Bennet J.": "Bennet J. Moskowitz",
    "NYTimes.com News Alert": "NYTimes.com (News Alert)",
    "Neal Kassel": "Neal Kassell",
    "Neil Gershenfeld <>": "Neil Gershenfeld",
    "Newsmax. corn": "Newsmax",
    "Niall McCarthy | Statista": "Niall McCarthy",
    "Nowak, Martin A.": "Martin A. Nowak",
    "Ozan Tarman <_____________________": "Ozan Tarman",
    "PETER MANDELSON": "Peter Mandelson",
    "Peter Mandelson <________________________________": "Peter Mandelson",
    "PLI": "Desconhecido 14",
    "Paula": "Desconhecido (Paula)",
    "Palazzolo, Joseph": "Joseph Palazzolo",
    "Parker, Ashley": "Ashley Parker",
    "Parker, Ashley <>": "Ashley Parker",
    "Peggy": "Peggy Siegal",
    "Peggy Siegal <>": "Peggy Siegal",
    "Peggy SiegalliM1111.1.1": "Peggy Siegal",
    "Peter Atti": "Peter Attia",
    "Pickier, Nedra": "Nedra Pickier",
    "Pizzurro, Frank LA": "Frank Pizzurro",
    "Prince Bandar bin Sultan": "Principe Bandar Bin Sultan",
    "Pritzker, Tom": "Tom Pritzker",
    "Nadia": "Desconhecido (Nadia)",
    "Renata B": "Renata Bolotova",
    "Richard Kahn <>": "Richard Kahn",
    "Richard Kahn___________________________": "Richard Kahn",
    "Robert Grusky <>": "Robert Grusky",
    "Robert Kuhn": "Robert Lawrence Kuhn",
    "Robert Kuhn •": "Robert Lawrence Kuhn",
    "Robert L. Kuhn": "Robert Lawrence Kuhn",
    "Robert Trivers <>": "Robert Trivers",
    "Robert Triversi": "Robert Trivers",
    "Rush, George": "George Rush",
    "Rush, George mailto": "George Rush",
    "Scott J. Lin": "Scott J. Link",
    "Sean Banno": "Sean Bannon",
    "Sharon Churcher <>": "Sharon Churcher",
    "Starr, Ken": "Ken Starr",
    "Stephen Hanso": "Stephen Hanson",
    "Steve Bannon <": "Steve Bannon",
    "Steve Bannon <>": "Steve Bannon",
    "Steve Bannon <__________________": "Steve Bannon",
    "Steve Bannon <____________________": "Steve Bannon",
    "Steve Bannor": "Steve Bannon",
    "Steve Hanson <>": "Steve Hanson",
    "Steven Elkman <>": "Steven Elkman",
    "Steven Pfeiffer <_": "Steven Pfeiffer",
    "Steven Pfeiffer <_________________": "Steven Pfeiffer",
    "Stevenhoffenberg": "Steven Hoffenberg",
    "Sullivan, John": "John Sullivan",
    "Tanzi, Rudolph E": "Rudolph E. Tanzi",
    "Thomas Jr., Landon": "Landon Thomas Jr.",
    "homas Jr., Landon <>": "Landon Thomas Jr.",
    "Thomas Jr., Landon <_______________________": "Landon Thomas Jr.",
    "Thomas Jr., Landon <________________________": "Landon Thomas Jr.",
    "Thomas Jr., Landor": "Landon Thomas Jr.",
    "Thomas Jr., Landon <>": "Landon Thomas Jr.",
    "Thorbjon JagIan": "Thorbjørn Jagland",
    "Thorbjon Jagland": "Thorbjørn Jagland",
    "Thorbjon Jagland <>": "Thorbjørn Jagland",
    "Tonja Haddad Coleman <>": "Tonja Haddad Coleman",
    "Vincenzo lozzo": "Vincenzo Lozzo",
    "Weingarten, Reid": "Reid Weingarten",
    "Wolfe, Alexandra": "Alexandra Wolfe",
    "Zimmerman, Malia McLaughlin": "Malia McLaughlin Zimmerman",
    "abuulabass": "Desconhecido (abuulabass)",
    "admin": "Desconhecido (admin)",
    "anasalrasheed": "Anas Al-Rasheed",
    "anasalrasheed1111": "Anas Al-Rasheed",
    "anasalrasheed______________": "Anas Al-Rasheed",
    "aziza alahmadi": "Aziza Alahmadi",
    "drsra": "Desconhecido (drsra)",
    "ed thompson": "Ed Thompson",
    "ehbarak": "Ehud Barak",
    "elisabeth feliho": "Elisabeth Feliho",
    "habebey": "Desconhecido (habebey)",
    "lawkrauss": "Lawrence Krauss",
    "live:linkspirit": "Live: linkspirit",
    "martin Weinberg": "Martin Weinberg",
    "nk.mb": "Desconhecido (nk.mb)",
    "paul krassner": "Paul Krassner",
    "ro": "Desconhecido (ro)",
    "sender name or email": "Desconhecido (sender name or email)",
    "soon yi previn": "Soon-Yi Previn",
    "soon yi previn_________________________": "Soon-Yi Previn",
    "stanpottinger": "Stanley Pottinger",
    "stanpottinger mailto": "Stanley Pottinger",
    "steven hoffenberg": "Steven Hoffenberg",
    "Shaher":"Desconhecido (Shaher)",
    "Sheikh":"Desconhecido (Sheikh)",
    "tamem": "Desconhecido (tamem)",
    "thnx jeff i m praying44": "Desconhecido (thnx jeff i m praying44)"
}

df_saida = df_saida.copy()
df_saida.loc[:, 'de'] = df_saida['de'].replace(mapeamento)

In [28]:
pessoas_de = df_saida['de'].unique()
pessoas_de = sorted(pessoas_de)
for pessoa in pessoas_de:
    print(pessoa)

A Barrett
Afri zp
Al Franken
Al seckel
Alan Dershowitz
Alan Rogers
Alexandra Preate
Alexandra Wolfe
Alireza Ittihadieh
Amanda Ens
Anas Al-Rasheed
Anas Alrasheed
Anders Corr
Andres Serrano
Andrew Friendly
Anil Ambani
Ann Coulter
Annette Witheridge
Ariane de Rothschild
Ashley Parker
Auren Hoffman
Aziza Alahmadi
Barbro C. Ehnbom
Barnaby Marsh
Barry J. Cohen
Benjamin Harnwell
Benjamin Wegg-Prosser
Bennet J. Moskowitz
Bill Siegel
Blanche Christerso
Bloomberg
Bob Crowe
Boris Nikolic
Brad Edwards
Brad S. Karp
Brad Wechsler
Brandon Thompson
Brian Heberlig
Bruce Moskowitz
Bryan Subotnick
CLL
CNS Adam Klasfeld
Caroline Lang
Carolyn Rangel
Cathy Alexander
Cecilia Schött
Cecilia Steen
Charles Michael
Christina Galbraith
Comms Alert
Dan Christensen
Dangene and Jennie Enterprise
Daniel Sabba
Daniel Siad
Darren K. Indyke
Darren lndyke
Dave Hope
David Bianco, Deutsche Bank
David Blaine
David Fiszel
David Grosof
David Haig
David I. Schoen
David Ingram News
David Mitchell
David Pegg
David Schoen
David S

Agora olhando para a coluna 'para':

In [29]:
pessoas_para = df_saida['para'].unique()
pessoas_para = sorted(pessoas_para)
for pessoa in pessoas_para:
    print(pessoa)

#L&W BD PR US
'Cavan Mahony'
'Cavan mahony'
'Esneh'
'Jean'
'Martin G. Weinberg'
-11>
111110
<1111111
<MEMIN
ANN BAISE
AdeR
Adrienne Ross
Al seckel
Al seckel <
Alan Dershowitz
Alan Fraade
Alan M. Dershowitz
Alan M. Dershowitz Martin Weinberg
Alan Rogers
Alex
Alex LA
Alex.Family Pines
Alexander Marlow
Alexander Marlow <>
Alexandra
Alireza ITTIHADIEH
Alireza Ittihadieh
Allen West
Amanda
Anas Alrasheed
Andreas Kraus
Andres Serrano
Anna Dreber
Anthony
Anula Jayasuriya
Apparently
Ariane de Rothschild
Authoritarian Influence
Authoritarian Influence <MIM
Balin, Robert
Barb Cowles
Barbro C Ehnbom
Barbro Ehnbom
Barnaby Marsh
Barry J. Cohen
Barry J. Cohen <>
Barry Josephson
Bauer
Bauer, Steve SF-BR
Bella Klein
Belle Gray
Ben Sosenko
Benjamin Wegg-Prosser
Bennett Schmidt
Bill Prezant
Bill Siegel
Bob & Sandy
Bob Crowe
Bob Fass
Bobby McCormick
Boris Nikolic
Brad Edwards
Brad S
Brad S <___________________________
Brad S Karp
Brad S Karp__________________________
Brad Wechsler
Bradley J. Edwards
Brand

Vamos aplicar também um mapeamento específico, mas antes disso vamos tentar reutilizar o anterior.

In [30]:
df_saida = df_saida.copy()
df_saida.loc[:, 'para'] = df_saida['para'].replace(mapeamento)

In [31]:
pessoas_para = df_saida['para'].unique()
pessoas_para = sorted(pessoas_para)
for pessoa in pessoas_para:
    print(pessoa)

#L&W BD PR US
'Cavan Mahony'
'Cavan mahony'
'Esneh'
'Jean'
'Martin G. Weinberg'
111110
<MEMIN
ANN BAISE
AdeR
Adrienne Ross
Al seckel
Al seckel <
Alan Dershowitz
Alan Fraade
Alan M. Dershowitz
Alan M. Dershowitz Martin Weinberg
Alan Rogers
Alex
Alex LA
Alex.Family Pines
Alexander Marlow
Alexander Marlow <>
Alexandra
Alireza Ittihadieh
Allen West
Amanda
Anas Al-Rasheed
Anas Alrasheed
Andreas Kraus
Andres Serrano
Anil Ambani
Anna Dreber
Anthony
Anula Jayasuriya
Apparently
Ariane de Rothschild
Authoritarian Influence
Authoritarian Influence <MIM
Balin, Robert
Barb Cowles
Barbro C. Ehnbom
Barnaby Marsh
Barry J. Cohen
Bauer
Bauer, Steve SF-BR
Bella Klein
Belle Gray
Ben Sosenko
Benjamin Wegg-Prosser
Bennett Schmidt
Bill Prezant
Bill Siegel
Bob & Sandy
Bob Crowe
Bob Fass
Bobby McCormick
Boris Nikolic
Brad Edwards
Brad S
Brad S <___________________________
Brad S Karp
Brad S Karp__________________________
Brad Wechsler
Bradley J. Edwards
Brandon Thompson
Brett D. Jaffe
Brian Baker
Bridges Hague

Aplicando o novo mapeamento:

In [56]:
mapeamento_para = {
    "#L&W BD PR US": "L&W Supply: Commercial Building and Construction Materials",
    "Cavan Mahony": "Cavan Mahony",
    "111110": "Desconhecido 15",
    "<MEMIN": "Desconhecido 13",
    "ANN BAISE": "Desconhecido (ANN BAISE)",
    "AdeR": "Desconhecido (AdeR)",
    "Al seckel": "Al Seckel",
    "Al seckel <": "Al Seckel",
    "Alan M. Dershowitz": "Alan Dershowitz",
    "Alan M. Dershowitz Martin Weinberg": "Alan Dershowitz",
    "Alex": "Alex LA",
    "Alex.Family Pines": "Alex (Family Pines)",
    "Alexander Marlow <>": "Alexander Marlow",
    "Alexandra": "Alexandra Wolfe",
    "Amanda": "Amanda Ens",
    "Anas Alrasheed": "Anas Al-Rasheed",
    "Apparently": "Desconhecido (Apparently)",
    "Authoritarian Influence": "Desconhecido (Authoritarian Influence)",
    "Authoritarian Influence <MIM": "Desconhecido (Authoritarian Influence)",
    "Balin, Robert": "Robert Balin",
    "Bauer, Steve SF-BR": "Steve Bauer",
    "Bauer": "Steve Bauer",
    "Brad S": "Brad S. Karp",
    "Brad S <___________________________": "Brad S. Karp",
    "Brad S Karp": "Brad S. Karp",
    "Brad S Karp__________________________": "Brad S. Karp",
    "Bruno, Nicole NY": "Nicole Bruno",
    "Caroline": "Caroline Lang",
    "Cartwright, Lachlan": "Lachlan Cartwright",
    "Charles": "Desconhecido (Charles)",
    "Cheves, Belle": "Belle Cheves",
    "DANNY GOLDBERG": "Danny Goldberg",
    "Daniel CC": "Desconhecido (Daniel CC)",
    "Darren": "Darren K. Indyke",
    "Darren Indyk": "Darren K. Indyke",
    "Darren K.": "Darren K. Indyke",
    "Darren lndyke": "Darren K. Indyke",
    "Dean OC": "Desconhecido (Dean OC)",
    "Dixon, Timothy": "Timothy Dixon",
    "Dlugash, Alan": "Alan Dlugash",
    "Edward B. Reynolds, Jr.": "Edward B. Reynolds Jr.",
    "Eisenberg": "Marshall E. Eisenberg",
    "Ens": "Amanda Ens",
    "Epstein Jeffrey": "Jeffrey Epstein",
    "Eric": "Eric Roth",
    "Esq.": "Desconhecido(Esq.)",
    "Forrest Miller - adj kearsarge": "Forrest Miller",
    "Francis": "Francis Derby",
    "Friend": "Friends and Family",
    "Fwd: T V": "Desconhecido 16",
    "G G": "Gary Gross",
    "GREG FARRELL BLOOMBERG/ NEWSROOM:": "Greg Farrell",
    "Gary H. BaisallillW>": "Gary H. Baise",
    "Gray, Alex": "Alex Gray",
    "Greenberg": "Jeffrey Greenberg",
    "Greg LA": "Desconhecido(Greg LA)",
    "Gwendolyn": "Gwendolyn Beck",
    "HOUSE OVERSIGHT 031039": "House Oversight",
    "Haig": "Desconhecido (Haig)",
    "Halligan": "Desconhecido (Halligan)",
    "Harris, Nicole CH": "Nicole Harris",
    "Harris": "Harris, Nicole CH",
    "Henry hortenstine": "Henry Hortenstine",
    "Hill": "Desconhecido (Hill)",
    "IIII": "Desconhecido 17",
    "Ihsoffice": "IHS Offices",
    "Ihsofficel": "IHS Offices",
    "Immelt": "Jeff Immelt",
    "Indyke": "Darren K. Indyke",
    "Inside Job 2010": "Inside Job (2010)",
    "JR Stambaugh": "Stambaugh Jr.",
    "Jabor Y.": "Jabor Y. Inewey",
    "Jamie": "Jamie Rosenwald",
    "Jean Luc Brunel": "Jean Luc Brune",
    "Jeffrey": "Desconhecido (Jeffrey)",
    "Jeffrey&j effreyep stein. org": "jeffreyepstein.org",
    "Jennings, Alex LA": "Alex Jennings",
    "Jennings": "Alex Jennings",
    "Jim": "Desconhecido (Jim)",
    "Joi": "Joichi Ito",
    "Joichi Ito": "Joichi Ito",
    "Joichi Joi Ito": "Joichi Ito",
    "Karp": "Brad S. Karp",
    "Kathy DC": "",
    "Kathy Ruemmler Darren lndyke Martin Weinberg": "Kathryn Ruemmler",
    "Kathy Ruemmler Darren lndyke • Martin Weinberg": "Kathryn Ruemmler",
    "Kathy Ruemmler Martin Weinberg Alan Dershowitz": "Kathryn Ruemmler",
    "Kathy Ruemmler f": "Kathryn Ruemmler",
    "Kathy Ruemmler__________________": "Kathryn Ruemmler",
    "Ken": "Ken Starr",
    "Kensington2": "Marc Kensington",
    "LHS": "Lawrence H. Summers",
    "Lajcak Miroslav/MINISTER/MZV": "Miroslav Lajčák",
    "Lajcak Miroslay/MINISTER/MZV": "Miroslav Lajčák",
    "Landon Thomas Jr": "Landon Thomas Jr.",
    "Landon Thomas<_____________________": "Landon Thomas Jr.",
    "Lang": "Jack Lang",
    "Larry Summers": "Lawrence H. Summers",
    "LHS _____________________________": "Lawrence H. Summers",
    "Lawrence": "Lawrence H. Summers",
    "Lawrence H.": "Lawrence H. Summers",
    "Lawrence H. <>": "Lawrence H. Summers",
    "Lawrence Henry Summers": "Lawrence H. Summers",
    "Lawrence Henry Summers <>": "Lawrence H. Summers",
    "Lawrence Summers '11": "Lawrence H. Summers",
    "Lesley Groff <": "Lesley Groff",
    "Lesley Groff MMIIII": "Lesley Groff",
    "Lesley Groffl": "Lesley Groff",
    "Leslie": "Lesley Groff",
    "Lilly Sanchez": "Lilly Ann Sanchez",
    "Linda PINTO''": "Linda Pinto",
    "LouellaRabuycll Januiz Banasiak": "Desconhecido (LouellaRabuycll Januiz Banasiak)",
    "Louis": "Desconhecido (Louis)",
    "Lisa albert": "Lisa Albert",
    "Lyn fontanilla": "Lyn Fontanilla",
    "MD": "Masha Drokova",
    "MMIIIIIMIMI": "Michael Wolff",
    "MII": "Desconhecido 18",
    "Mahle, Melissa": "Melissa Mahle",
    "Margie": "Desconhecido (Margie)",
    "Mark epstein": "Mark Epstein",
    "Martin A. <MIIMINI": "Martin A.",
    "Martin G Weinberg": "Martin G. Weinberg",
    "Martin Weinberg Esq": "Martin G. Weinberg",
    "Martin Weinberg dkiesq": "Martin G. Weinberg",
    "Martin Weinberg<__________________": "Martin G. Weinberg",
    "Maskin": "Eric Maskin",
    "Mayor": "Desconhecido (Mayor)",
    "Melanie Spinel!": "Melanie Spinella",
    "Melanie Spinella <>": "Melanie Spinella",
    "Michael": "Michael Braun",
    "Michael Braun <i": "Michael Braun",
    "Michael Woli": "Michael Wolff",
    "Miller, Glen": "Glen Miller",
    "Miller": "Glen Miller",
    "Mohebbi, Nima LA": "Nima Mohebbi",
    "Mohebbi": "Nima Mohebbi",
    "Moore, Wendy": "Wendy Moore",
    "Moore": "Wendy Moore",
    "Nadia": "Desconhecido (Nadia)",
    "Nate mcclain": "Nate Mcclain",
    "Neil anderson": "Neil Anderson",
    "Nicholas": "Nicholas Gahhos",
    "Nima": "Nima Mohebbi",
    "Nowak": "Martin A. Nowak",
    "Nicole CH": "Nicole Bruno",
    "PETER MANDELSON.": "Peter Mandelson",
    "Paralegal": "Desconhecido (Paralegal)",
    "Paul Morris/db/": "Paul Morris",
    "Paula": "Desconhecido (Paula)",
    "Peggy Siega": "Peggy Siegal",
    "Peter Mandelson BT": "Peter Mandelson",
    "Players2 .": "Desconhecido (Players2)",
    "Pritzker": "Tom Pritzker",
    "RALPH BERNSTEIN": "Ralph Bernstein",
    "Reid": "Reid Hoffman",
    "Rich Kahn IMMIMMIIIIMII;": "Rich Kahn",
    "Richard Kahn_______________________": "Richard Kahn",
    "Richard Kahn______________________________": "Richard Kahn",
    "Richerson, Peter J": "Peter J. Richerson",
    "Richman, Lawrence": "Lawrence Richman",
    "Richman": "Lawrence Richman",
    "Robert D. Critton Jr. • Jessica Cadwell": "Jessica Cadwell",
    "Robins, Greg LA": "Greg Robins",
    "Robins": "Greg Robins",
    "Ruemmler": "Kathryn Ruemmler",
    "Ruemmler, Kathy DC": "Kathryn Ruemmler",
    "Sam/Walli Leff": "Samuel Leff",
    "Schecter, Daniel CC": "Daniel Schecter",
    "Schecter": "Daniel Schecter",
    "Scott Stambauh": "Scott Stambaugh",
    "Seche, Stephen A.": "Stephen A. Seche",
    "Silverman, Nicholas": "Nicholas Silverman",
    "Silverman": "Nicholas Silverman",
    "Soon Yi Previn": "Soon-Yi Previn",
    "Soon-Yi": "Soon-Yi Previn",
    "Soon-Yi Previn": "Soon-Yi Previn",
    "Starr": "Ken Starr",
    "Steve Bannon I": "Steve Bannon",
    "Steve Bannon<": "Steve Bannon",
    "Steve Bannon<1____________________": "Steve Bannon",
    "Steve Bannon<>": "Steve Bannon",
    "Steve SF-BR": "Steve Bauer",
    "Subotnick Stuart 11111111111111111111>": "Subotnick Stuart",
    "Summers": "Lawrence H. Summers",
    "Thorbjon Jaglan": "Thorbjørn Jagland",
    "Tom": "Tom Barrack",
    "Tom Barrack Privat": "Tom Barrack",
    "Tom Barrack Private": "Tom Barrack",
    "Torla Haddad Coleman": "Tonja Haddad Coleman",
    "Unspecified": "Desconhecido 19",
    "Val sherman": "Val Sherman",
    "Vinit Sahni/db/": "Vinit Sahni",
    "Weingarten": "Reid Weingarten",
    "Wendy OC": "Wendy Moore",
    "Wine, Jamie NY": "Jamie Wine",
    "Wine": "Jamie Wine",
    "Wolfe": "Alexandra Wolfe",
    "___________________________": "Desconhecido 20",
    "____________________________": "Desconhecido 21",
    "_______________________________": "Desconhecido 22",
    "behnborn": "Desconhecido (behnborn)",
    "dkiesq": "Martin G. Weinberg",
    "ehud barak___________________________": "Ehud Barak",
    "jac": "Desconhecido (jac)",
    "janet kafka": "Janet Kafka",
    "jean.huguen": "Jean Huguen",
    "jeffrey E.": "Jeffrey Epstein",
    "john zouzelka-cell": "John Zouzelka",
    "lhsoffic": "LHS Office",
    "lhsoffice": "LHS Office",
    "michael": "Michael Wolff",
    "nate white": "Nate White",
    "nicholas.ribis": "Nicholas Ribis",
    "paul prosperi": "Paul Prosperi",
    "recipient name or email": "Desconhecido (recipient name or email)",
    "rita hortenstine": "Rita Hortenstine",
    "rote": "Desconhecido (rote)",
    "soon yi previ": "Soon-Yi Previn",
    "taal safdie": "Taal Safdie",
    "undisclosed-recipients:": "Desconhecido 23",
    "uri fouzailov": "Uri Fouzailov"
}

In [57]:
df_saida = df_saida.copy()
df_saida.loc[:, 'para'] = df_saida['para'].replace(mapeamento_para)

In [58]:
df_saida = df_saida.drop_duplicates()

In [59]:
df_saida.shape

(2208, 3)

Vamos gerar um json para utilizar numa visualização interativa.

In [62]:
import json

col_de = "de"
col_para = "para"
col_assunto = "assunto"

# Vértices.
grau_series = pd.concat([df_saida[col_de], df_saida[col_para]]).value_counts()

vertices = [
    {"id": pessoa, "grau": int(grau)}
    for pessoa, grau in grau_series.items()
]

# Arestas.
arestas = [
    {
        "remetente": row[col_de],
        "destinatario": row[col_para],
        "assunto": row[col_assunto]
    }
    for _, row in df_saida.iterrows()
]

# Salvar em Json.
with open("grafo_emails.js", "w", encoding="utf-8") as f:
    f.write("const grafo = ")
    json.dump({"vertices": vertices, "arestas": arestas}, f, ensure_ascii=False, indent=2)
    f.write(";")